# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step workflow for loading and exploring the FAIR<sup>2</sup> Mass Spectrometry dataset using the `mlcroissant` library—directly from its Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

We'll inspect, extract, and process the dataset using its `@id` definitions for robustness and reproducibility.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\nIdentifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview

Review available record sets, fields, and their IDs. We'll list every record set `@id` present and the fields or columns of each. All IDs are referenced by their `@id`.

In [ ]:
# List all record sets and their fields using @id references
from pprint import pprint

# The Croissant JSON schema metadata exposes record sets as a list on metadata.record_sets
record_sets = getattr(metadata, 'record_sets', [])
if not record_sets:
    print("No record sets defined in the schema.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs['@id']}")
        # Each record set has fields (column definitions)
        # List all fields by their @id and name if available
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        if not fields:
            print("  (No fields defined)")
        else:
            for field in fields:
                label = field.get('name', field.get('@id', '(no name)'))
                print(f"  Field: {field['@id']} (name: {label})")

### (Optional) Explore records structure
Display the first record from a chosen record set by its `@id`:

In [ ]:
# List a sample record for the first available record set
if record_sets:
    first_record_set_id = record_sets[0]['@id']
    sample = next(dataset.records(record_set=first_record_set_id), None)
    print(f"Sample record from record set {first_record_set_id}:")
    pprint(sample)
else:
    print("No record sets available for sampling.")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. All entity references (record sets, fields/columns) use their `@id` as per the Croissant schema. We'll extract all record sets and prepare them for downstream processing.

In [ ]:
# Extract data from each record set into DataFrames
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets] if record_sets else []

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Record set {record_set_id}: loaded {df.shape[0]} rows and {df.shape[1]} columns.")
        else:
            print(f"Record set {record_set_id} contains no data.")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

if dataframes:
    # Show columns of the first available DataFrame
    sample_record_set_id = list(dataframes.keys())[0]
    print(f"First DataFrame columns ({sample_record_set_id}):")
    print(dataframes[sample_record_set_id].columns.tolist())
    display(dataframes[sample_record_set_id].head())
else:
    print("No dataframes were loaded from the dataset.")

## 4. Exploratory Data Analysis (EDA)

Let's process a numeric field from the main record set. We'll filter, normalize, and group as follows:
- Select a numeric field by its `@id` (or column name, which in Croissant is usually the `@id`)
- Filter for records above a threshold
- Normalize the numeric column
- Group by a key attribute if present

Replace the example `numeric_field_id` and `group_field_id` with real ones from your schema/step 2 above if running interactively.

In [ ]:
# Edit these to valid @ids from your dataset, or choose automatically:
if dataframes:
    df = dataframes[sample_record_set_id]
    # Try automatic detection: pick the first numeric column
    import numpy as np
    numeric_candidates = df.select_dtypes(include=[np.number]).columns
    if len(numeric_candidates):
        numeric_field_id = numeric_candidates[0]
    else:
        # Try finding something that looks like a numeric column
        for col in df.columns:
            try:
                df[col] = df[col].astype(float)
                numeric_field_id = col
                break
            except Exception:
                continue
        else:
            numeric_field_id = None
            print("No numeric field detected.")

    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype in [np.float64, np.int64] else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Automatic grouping by a string-like field (choose the first object column other than the numeric)
        group_candidates = [c for c in df.columns if c != numeric_field_id and df[c].dtype == object]
        if group_candidates:
            group_field_id = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found in the DataFrame for analysis.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization

Visualize distributions or relationships between fields in the main record set. We'll plot a histogram for the main numeric field and, if possible, a boxplot grouped by the chosen key.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # Boxplot by group if group_field present
    if 'group_field_id' in locals():
        if group_field_id in df.columns:
            plt.figure(figsize=(10,4))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
            plt.title(f"{numeric_field_id} grouped by {group_field_id}")
            plt.show()
else:
    print("Cannot plot: No numeric field detected or DataFrame missing.")

## 6. Conclusion

- We demonstrated loading Croissant metadata and all records using `mlcroissant`.
- We explored the structure, selected fields by their `@id`, filtered and normalized a main numeric field, and visualized data distributions.
- For best reproducibility, always reference dataset entities (record sets, fields, columns) using their Croissant `@id`.

Continue by building domain-specific analytic workflows: e.g., protein abundance comparisons, peptide count summaries, etc., using the processed DataFrames. For further help, see the `mlcroissant.Dataset` documentation.